In [25]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import base64
import requests
import numpy
import tqdm 

# Load API key from .env
load_dotenv()
api_key=os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def divide_into_batches(array, batch_size):
    no_batches = int(numpy.ceil(len(array) / batch_size))
    batches = []

    for b in range(no_batches):
        batches += [array[b * batch_size : ((b + 1) * batch_size)]]
    
    return batches

In [26]:
path = f'/home/{os.getlogin()}/Pictures/Screenshots/'
image_paths = [path + file_name for file_name in os.listdir(path) if file_name[-4:] == '.png']
already_done = []

with open("already_done.txt", "r", encoding="utf-8") as f:
    already_done += f.read().strip().splitlines()

image_paths = [path for path in image_paths if path not in already_done]
print(image_paths)

with open("already_done.txt", "a", encoding="utf-8") as f:
    for img in image_paths:
        f.write(img + "\n")

base64_images = [encode_image(img_path) for img_path in image_paths]
b64_batches = divide_into_batches(base64_images, batch_size = 3)
print([len(b) for b in b64_batches])

['/home/wiksusz/Pictures/Screenshots/Screenshot From 2025-06-25 15-16-28.png', '/home/wiksusz/Pictures/Screenshots/Screenshot From 2025-06-25 15-16-43.png', '/home/wiksusz/Pictures/Screenshots/Screenshot From 2025-06-25 15-22-32.png', '/home/wiksusz/Pictures/Screenshots/Screenshot From 2025-06-25 15-22-48.png', '/home/wiksusz/Pictures/Screenshots/Screenshot From 2025-06-25 15-22-58.png']
[3, 2]


In [27]:
for batch in tqdm.tqdm(b64_batches):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}"
    }

    content_list = [
        {
            "type": "image_url",
            "image_url": {
            "url": f"data:image/jpeg;base64,{base64_image}"
            }
        } for base64_image in batch
    ]

    content_list = [
            {
                "type": "text",
                "text": "What’s the text in this images? Mark chosen answer with ✔️."
            }
    ] + content_list

    payload = {
        "model": "gpt-4.1",
        "messages": [
        {
            "role": "user",
            "content": content_list 
        }
        ],
        "max_tokens": 2000 * len(batch)
    }

    response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)
    print(f"{response.json()}"[:100])

    response_text = response.json()['choices'][0]['message']['content']
    with open("text.txt", "a", encoding="utf-8") as f:
            f.write(response_text)

 50%|█████     | 1/2 [00:21<00:21, 21.77s/it]

{'id': 'chatcmpl-BmNJ4gbEvtUqAkqWZtlv8Ku7RTHlR', 'object': 'chat.completion', 'created': 1750869034,


100%|██████████| 2/2 [00:34<00:00, 17.29s/it]

{'id': 'chatcmpl-BmNJOz5wg4eE9Rd1UbgCVBw8sl0rp', 'object': 'chat.completion', 'created': 1750869054,
